# Cluster intra-RMSD analysis

Compare structure variation within `cluster_ids` groups for ground truth (pkl) and representative predictions, and relate both to prediction-vs-GT RMSD.

This notebook produced figures 3.1a,b, and 3.5a. This notebook also has the computation of the stats reported in the first paragraph of the 'Variance of Predictions' section

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_theme(style="whitegrid")

BASE = Path("/opig-shared/users/lina4783/abb4_experiments/evaluation/cluster_intra_rmsd")
OUT_DIR = BASE / "outputs"
FIG_DIR = BASE / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = Path(
    "/opig-shared/users/lina4783/abb4_experiments/evaluation/predictions_ckpt_5139_imgt/struc_pred_metrics_test_summary.csv"
)
#SUMMARY_PATH = Path(
#    "/opig-shared/users/lina4783/abb4_experiments/evaluation/predictions_nonsubsample_imgt/struc_pred_metrics_test_summary.csv"
#)
#ALL_SAMPLES_PATH = Path(
#    "/opig-shared/users/lina4783/abb4_experiments/evaluation/predictions_sample_imgt/struc_pred_metrics_test_all_samples.csv"
#)
ALL_SAMPLES_PATH = Path(
    "/opig-shared/users/lina4783/abb4_experiments/evaluation/predictions_ckpt_5139_imgt/struc_pred_metrics_test_all_samples.csv"
)

TEST_META_PATH = Path("/opig-shared/users/lina4783/structures_final/test_meta.csv")

PRIMARY_METRICS = ["H_cdr3", "H_cdr_all", "L_cdr_all", "L_cdr3"]

In [ ]:
# Intra-cluster means: same-length CDR pairs only (per metric; pair_length_filters.py).
#make summary df of gt rmsd, pred rmsd, and pred wrt gt rmsd for each cluster

gt = pd.read_csv(OUT_DIR / "cluster_gt_pairwise_rmsd_same_len.csv")
pred = pd.read_csv(OUT_DIR / "ckpt_5139/cluster_pred_pairwise_rmsd_same_len.csv")
summary = pd.read_csv(SUMMARY_PATH)
all_samples = pd.read_csv(ALL_SAMPLES_PATH)
test_meta = pd.read_csv(TEST_META_PATH, index_col=0)

summary_ok = summary[summary["status"] == "ok"].copy()
summary_ok = summary_ok.merge(
    test_meta[["cluster_ids","pdb_name"]].reset_index(), on="pdb_name", how="left"
)

all_samples = all_samples.merge(
    test_meta[["cluster_ids","pdb_name"]].reset_index(), on="pdb_name", how="left"
)

cluster_pred_vs_gt = (
    summary_ok.groupby("cluster_ids")[PRIMARY_METRICS]
    .median()
    .add_prefix("pred_vs_gt_median_")
    .reset_index()
)

cluster_sample_spread = (
    all_samples.groupby("cluster_ids")[PRIMARY_METRICS]
    .agg(["mean", "std"])
)

cluster_sample_spread.columns = [
    f"all_samples_{metric}_{stat}" for metric, stat in cluster_sample_spread.columns
]
cluster_sample_spread = cluster_sample_spread.reset_index()

gt_cols = {f"mean_pairwise_{m}": f"gt_intra_{m}" for m in PRIMARY_METRICS}
pred_cols = {f"mean_pairwise_{m}": f"pred_intra_{m}" for m in PRIMARY_METRICS}

#keeps all gt columns, pred cols has only PRIMARY_METRICS
merged = gt.rename(columns=gt_cols).merge(
    pred.rename(columns=pred_cols)[["cluster_ids"] + list(pred_cols.values())],
    on="cluster_ids",
    how="inner",
)
merged = merged.merge(cluster_pred_vs_gt, on="cluster_ids", how="left")
merged = merged.merge(cluster_sample_spread, on="cluster_ids", how="left")

#merged.to_csv(OUT_DIR / "cluster_rmsd_merged.csv", index=False)
print(f"Merged table: {len(merged)} clusters with both GT and pred intra-cluster RMSD")
same_len_cols = [c for c in gt.columns if c.startswith("n_pairs_computed_")]
if same_len_cols:
    print("Median n_pairs_computed_* per cluster (same-length filtered):")
    display(gt[same_len_cols].median().to_frame("median_pairs").astype(int))
merged.head()

In [ ]:
merged[merged.cluster_ids==3][["cluster_ids","gt_intra_H_cdr3","pred_intra_H_cdr3","pred_vs_gt_median_H_cdr3"]]


In [ ]:
#TODO: find a cluster of size 2 with high pairwise rmsdand visualize gt structures in pymol
merged[merged.gt_intra_H_cdr3<0][["cluster_ids","gt_intra_H_cdr3","pred_intra_H_cdr3","pred_vs_gt_median_H_cdr3"]]

## 2a — Distribution of intra-cluster RMSDs

Cluster means use **same-length CDR pairs only** (per metric; e.g. equal `CDRH3` length for `H_cdr3`).


In [ ]:
# Pairwise GT H CDR3 RMSDs pooled by cluster size
gt_pairs = pd.read_csv(OUT_DIR / "cluster_gt_pairwise_pairs.csv")
pair_df = (
    gt_pairs.merge(gt[["cluster_ids", "n_members"]], on="cluster_ids", how="left")
    .query("status == 'ok'")
    .dropna(subset=["H_cdr3", "n_members"])
)
size_order = sorted(pair_df["n_members"].unique())
n_clusters = (
    pair_df.groupby("n_members")["cluster_ids"]
    .nunique()
    .reindex(size_order)
)

from matplotlib.transforms import blended_transform_factory

COUNT_COLOR = "#87CEEB"

fig, ax = plt.subplots(figsize=(14, 5))
sns.violinplot(
    data=pair_df,
    x="n_members",
    y="H_cdr3",
    order=size_order,
    density_norm="width",
    cut=0,
    inner="quartile",
    ax=ax,
    color="#4C72B0",
)
ax.set_xticks(range(len(size_order)))
ax.set_xticklabels([f"{size}\n{int(n_clusters[size])}" for size in size_order])
ax.set_title("Pairwise CDRH3 RMSD between GT Structures")
ax.set_xlabel("Cluster size (n members)\n# of clusters")
ax.set_ylabel("Pairwise CDRH3 RMSD (Å)")
fig.subplots_adjust(bottom=0.18)
fig.tight_layout()
fig.savefig(FIG_DIR / "gt_pairwise_H_cdr3_violin_by_cluster_size.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"{len(pair_df):,} pairwise comparisons across {pair_df['n_members'].nunique()} cluster sizes")
pair_df.groupby("n_members").agg(
    n_clusters=("cluster_ids", "nunique"),
    n_pairs=("H_cdr3", "count"),
    median_H_cdr3=("H_cdr3", "median"),
    mean_H_cdr3=("H_cdr3", "mean"),
).sort_index()

In [ ]:
def plot_intra_distributions(metric="H_cdr3"):
    gt_col = f"gt_intra_{metric}"
    plot_df = merged[[gt_col, "n_members"]].dropna()

    fig, ax = plt.subplots(figsize=(6, 4))

    sns.histplot(
        data=plot_df,
        x=gt_col,
        kde=True,
        ax=ax,
        element="step"
    )

    ax.set_title(f"Distribution of mean pairwise {metric} RMSD (same-length pairs) within clusters")
    ax.set_xlabel("Mean pairwise RMSD (Å)")

    fig.tight_layout()
    fig.savefig(FIG_DIR / f"intra_distribution_{metric}_gt_only.png", dpi=150, bbox_inches="tight")
    plt.show()


for metric in PRIMARY_METRICS:
    plot_intra_distributions(metric)

In [ ]:
def plot_intra_distributions(metric="H_cdr3"):
    gt_col = f"gt_intra_{metric}"
    pred_col = f"pred_intra_{metric}"
    plot_df = merged[[gt_col, pred_col, "n_members"]].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    long = plot_df.melt(value_vars=[gt_col, pred_col], var_name="source", value_name="rmsd")
    long["source"] = long["source"].map({gt_col: "Ground truth", pred_col: "Prediction"})
    sns.histplot(data=long, x="rmsd", hue="source", kde=True, ax=axes[0], element="step")
    axes[0].set_title(f"Distribution of mean pairwise intra-cluster {metric} (same-length pairs)")
    axes[0].set_xlabel("Mean pairwise RMSD (Å)")

    sns.boxplot(data=long, x="source", y="rmsd", ax=axes[1])
    axes[1].set_title(f"{metric} RMSD Distribution")
    axes[1].set_xlabel("")

    fig.tight_layout()
    fig.savefig(FIG_DIR / f"intra_distribution_{metric}.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=plot_df,x="n_members",y=gt_col,ax=ax,color="#4C72B0")
    ax.set_title(f"GT intra-cluster {metric} by cluster size")
    ax.set_xlabel("Cluster size (n members)")
    ax.set_ylabel("Mean pairwise RMSD (Å)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"gt_intra_by_cluster_size_{metric}.png", dpi=150, bbox_inches="tight")
    plt.show()

for metric in PRIMARY_METRICS:
    plot_intra_distributions(metric)

## 2b — GT vs prediction intra-cluster variation

Same-length pair filtering as in 2a.


In [ ]:
def correlation_report(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    pearson_r, pearson_p = stats.pearsonr(x, y)
    spearman_r, spearman_p = stats.spearmanr(x, y)
    return {
        "n": len(x),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_r": spearman_r,
        "spearman_p": spearman_p,
    }

corr_rows = []
for metric in PRIMARY_METRICS:
    gt_col = f"gt_intra_{metric}"
    pred_col = f"pred_intra_{metric}"
    stats_dict = correlation_report(merged[gt_col].values, merged[pred_col].values)
    stats_dict["metric"] = metric
    corr_rows.append(stats_dict)

    fig, ax = plt.subplots(figsize=(5, 5))
    plot_df = merged[[gt_col, pred_col]].dropna()
    ax.scatter(plot_df[gt_col], plot_df[pred_col], alpha=0.6, s=25)
    lims = [
        min(plot_df[gt_col].min(), plot_df[pred_col].min()),
        max(plot_df[gt_col].max(), plot_df[pred_col].max()),
    ]
    ax.plot(lims, lims, "k--", linewidth=1)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel(f"GT mean pairwise {metric} (Å)")
    ax.set_ylabel(f"Pred mean pairwise {metric} (Å)")
    ax.set_title(
        f"GT vs pred intra-cluster {metric}\n"
        f"Spearman r={stats_dict['spearman_r']:.3f}, p={stats_dict['spearman_p']:.2e}, n={stats_dict['n']}"
    )
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"gt_vs_pred_intra_{metric}.png", dpi=150, bbox_inches="tight")
    plt.show()

corr_df = pd.DataFrame(corr_rows)
corr_df

## 2c — Intra-cluster variation vs prediction accuracy

Same-length pair filtering as in 2a.


In [ ]:
metric = "H_cdr3"
cols = [
    f"gt_intra_{metric}",
    f"pred_intra_{metric}",
    f"pred_vs_gt_median_{metric}",
]
analysis_df = merged[cols + ["n_members"]].dropna()

pairplot = sns.pairplot(
    analysis_df.rename(columns={
        cols[0]: "GT intra-cluster",
        cols[1]: "Pred intra-cluster",
        cols[2]: "Pred vs GT (median)",
    }),
    diag_kind="hist",
    corner=True,
    plot_kws={"alpha": 0.5, "s": 20},
)
pairplot.figure.savefig(FIG_DIR / f"pairplot_intra_vs_accuracy_{metric}.png", dpi=150, bbox_inches="tight")
plt.show()

relationship_rows = []
target = f"pred_vs_gt_median_{metric}"
for source_col, label in [
    (f"gt_intra_{metric}", "gt_intra"),
    (f"pred_intra_{metric}", "pred_intra"),
]:
    sub = analysis_df[[source_col, target]].dropna()
    r, p = stats.spearmanr(sub[source_col], sub[target])
    relationship_rows.append({"predictor": label, "target": target, "spearman_r": r, "p_value": p, "n": len(sub)})

relationship_df = pd.DataFrame(relationship_rows)
relationship_df

In [ ]:
# Optional: multiple regression with cluster size as covariate
import statsmodels.formula.api as smf

reg_df = merged[
    [f"gt_intra_{metric}", f"pred_intra_{metric}", f"pred_vs_gt_median_{metric}", "n_members"]
].dropna()

if len(reg_df) >= 10:
    model = smf.ols(
        f"pred_vs_gt_median_{metric} ~ gt_intra_{metric} + pred_intra_{metric} + n_members",
        data=reg_df,
    ).fit()
    print(model.summary())
else:
    print("Not enough clusters for regression.")

## Combined train+test: GT pairwise H CDR3 by cluster size (with reference bounds)

Requires train GT pairwise outputs under `outputs/train/`. **Left:** pairwise `H_cdr3` among structures with identical `concat_CDR` (from existing pair CSVs joined to `metadata_no_nano.csv`; pairs exist only within the same `cluster_ids`). **Right:** upper bound from `outputs/bounds/global_subsample_H_cdr3_pairs.csv` (50 replicates, k=80 random subsamples of the full non-nanobody set; run `compute_global_subsample_bounds.py` or `run_compute_global_subsample_bounds.sbatch` once). Middle violins and terracotta Pareto curve (cumulative **% of structures** by cluster size) are unchanged; Pareto is drawn only over cluster-size categories, not the two reference violins.


In [ ]:
TRAIN_OUT = OUT_DIR / "train"
BOUNDS_DIR = OUT_DIR / "bounds"
META_NO_NANO = Path("/opig-shared/users/lina4783/structures_final/metadata_no_nano.csv")

gt_pairs = pd.concat(
    [
        pd.read_csv(OUT_DIR / "cluster_gt_pairwise_pairs.csv", low_memory=False),
        pd.read_csv(TRAIN_OUT / "cluster_gt_train_pairwise_pairs.csv", low_memory=False),
    ],
    ignore_index=True,
)
gt_cluster = pd.concat(
    [
        pd.read_csv(OUT_DIR / "cluster_gt_pairwise_rmsd.csv")[["cluster_ids", "n_members"]],
        pd.read_csv(TRAIN_OUT / "cluster_gt_train_pairwise_rmsd.csv")[["cluster_ids", "n_members"]],
    ],
    ignore_index=True,
)

meta = pd.read_csv(META_NO_NANO)
cdr_n_clusters = meta.groupby("concat_CDR")["cluster_ids"].nunique()
print(
    f"concat_CDR values spanning >1 cluster_id: {(cdr_n_clusters > 1).sum()} "
    "(lower bound only includes identical-CDR pairs within the same cluster)"
)

pdb_to_cdr = meta.set_index("pdb_name")["concat_CDR"]
pairs_cdr = gt_pairs.assign(
    concat_CDR_a=gt_pairs["pdb_name_a"].map(pdb_to_cdr),
    concat_CDR_b=gt_pairs["pdb_name_b"].map(pdb_to_cdr),
)
lower_bound = pairs_cdr.query(
    "status == 'ok' and concat_CDR_a == concat_CDR_b and concat_CDR_a.notna()"
)["H_cdr3"]

BOUNDS_DIR.mkdir(parents=True, exist_ok=True)
lower_bound.to_frame("H_cdr3").to_csv(BOUNDS_DIR / "identical_concat_cdr_H_cdr3.csv", index=False)

pair_df = (
    gt_pairs.merge(gt_cluster, on="cluster_ids", how="left")
    .query("status == 'ok'")
    .dropna(subset=["H_cdr3", "n_members"])
)
pair_df["plot_group"] = pair_df["n_members"].astype(int).astype(str)

upper_path = BOUNDS_DIR / "global_subsample_H_cdr3_pairs.csv"
if upper_path.is_file():
    upper_bound = (
        pd.read_csv(upper_path).query("status == 'ok'").dropna(subset=["H_cdr3"])["H_cdr3"]
    )
else:
    upper_bound = pd.Series(dtype=float)
    print(f"Warning: {upper_path} not found — run compute_global_subsample_bounds.py for the upper-bound violin")

LOWER_LABEL = "Identical CDR seq"
UPPER_LABEL = "Global subsample"
plot_all = pd.concat(
    [
        pd.DataFrame({"H_cdr3": lower_bound, "plot_group": LOWER_LABEL}),
        pair_df[["H_cdr3", "plot_group"]],
        pd.DataFrame({"H_cdr3": upper_bound, "plot_group": UPPER_LABEL}),
    ],
    ignore_index=True,
)

size_order = sorted(pair_df["n_members"].unique())
group_order = [LOWER_LABEL] + [str(int(s)) for s in size_order] + [UPPER_LABEL]

palette = {LOWER_LABEL: "#6B9080", UPPER_LABEL: "#7D6B7D"}
palette.update({str(int(s)): "#5F7D95" for s in size_order})

total_structures = int(gt_cluster["n_members"].sum())
n_groups = len(group_order)
n_middle = len(size_order)
x_pos_middle = list(range(1, 1 + n_middle))
cum_pct = [
    gt_cluster.loc[gt_cluster["n_members"] <= s, "n_members"].sum() / total_structures * 100
    for s in size_order
]

MIDDLE_COLOR = "#5F7D95"
CUM_COLOR = "#A65D57"

fig_w = max(18, n_groups * 0.55)
fig, ax = plt.subplots(figsize=(fig_w, 5.5))
sns.violinplot(
    data=plot_all,
    x="plot_group",
    y="H_cdr3",
    order=group_order,
    hue="plot_group",
    palette=palette,
    density_norm="width",
    cut=0,
    inner="quartile",
    width=0.55,
    ax=ax,
    dodge=False,
    legend=False,
)
ax.set_xlim(-0.75, n_groups - 0.25)
ax.set_xticks(range(n_groups))
ax.set_xticklabels(group_order, rotation=45, ha="right")
ax.set_xlabel("Lower reference / cluster size (n) / upper reference")
ax.set_ylabel("Pairwise CDRH3 RMSD (Å)", color=MIDDLE_COLOR)
ax.tick_params(axis="y", labelcolor=MIDDLE_COLOR)
ax.spines["left"].set_color(MIDDLE_COLOR)
ax.set_title("Intra-cluster Structural Diversity of CDRH3 (with reference bounds)")

ax2 = ax.twinx()
ax2.plot(x_pos_middle, cum_pct, color=CUM_COLOR, marker="o", linewidth=2, markersize=4)
for x, y in zip(x_pos_middle, cum_pct):
    ax2.annotate(
        f"{y:.1f}%",
        (x, y),
        textcoords="offset points",
        xytext=(0, 6),
        ha="center",
        va="bottom",
        fontsize=9,
        color=CUM_COLOR,
    )
ax2.set_ylabel("Cumulative % of structures (by cluster size)", color=CUM_COLOR)
ax2.tick_params(axis="y", labelcolor=CUM_COLOR)
ax2.spines["right"].set_color(CUM_COLOR)
ax2.set_ylim(0, max(100, max(cum_pct) * 1.08))
ax2.set_xlim(ax.get_xlim())

fig.tight_layout()
fig.savefig(FIG_DIR / "gt_pairwise_H_cdr3_violin_combined_pareto_with_bounds.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    f"Middle: {len(pair_df):,} intra-cluster pairs; lower: {len(lower_bound):,} identical-CDR pairs; "
    f"upper: {len(upper_bound):,} subsampled global pairs; "
    f"{total_structures:,} structures in {len(gt_cluster):,} clusters"
)
pd.DataFrame({"n_members": size_order, "cum_pct_structures": cum_pct}).set_index("n_members")


## CDRH3 sequence length vs intra-cluster GT H_cdr3 (train + test)

Ground-truth pairwise **H_cdr3** within the same `cluster_ids`, combined over **test** ([`cluster_gt_pairwise_pairs.csv`](outputs/cluster_gt_pairwise_pairs.csv)) and **train** ([`outputs/train/cluster_gt_train_pairwise_pairs.csv`](outputs/train/cluster_gt_train_pairwise_pairs.csv)), with CDRH3 lengths from [`test_meta.csv`](../../structures_final/test_meta.csv) and [`train_meta.csv`](../../structures_final/train_meta.csv).

RMSD uses framework-aligned IMGT CDRH3 (same as the rest of this notebook). Length is ** amino-acid length of the CDRH3 sequence**, not gap-aligned length.

Figures: (1) RMSD vs length mismatch, (2) same-length vs mixed-length pairs, (3) sparsity-masked cluster × mean-pair-length heatmap, (4) pooled length × length heatmap, (5) cluster-level length heterogeneity vs mean pairwise H_cdr3.


In [ ]:
TRAIN_OUT = OUT_DIR / "train"
TRAIN_META_PATH = Path("/opig-shared/users/lina4783/structures_final/train_meta.csv")

MIN_PAIRS_HEATMAP = 8
TOP_CLUSTERS_FOR_HEATMAP = 25
PRIMARY_PAIR_METRIC = "H_cdr3"

gt_pairs_all = pd.concat(
    [
        pd.read_csv(OUT_DIR / "cluster_gt_pairwise_pairs.csv", low_memory=False).assign(split="test"),
        pd.read_csv(TRAIN_OUT / "cluster_gt_train_pairwise_pairs.csv", low_memory=False).assign(split="train"),
    ],
    ignore_index=True,
)
gt_cluster_all = pd.concat(
    [
        pd.read_csv(OUT_DIR / "cluster_gt_pairwise_rmsd.csv")[["cluster_ids", "n_members"]],
        pd.read_csv(TRAIN_OUT / "cluster_gt_train_pairwise_rmsd.csv")[["cluster_ids", "n_members"]],
    ],
    ignore_index=True,
)
gt_cluster_all = gt_cluster_all.groupby("cluster_ids", as_index=False)["n_members"].max()

test_meta_len = pd.read_csv(TEST_META_PATH, usecols=["pdb_name", "CDRH3"])
train_meta_len = pd.read_csv(TRAIN_META_PATH, usecols=["pdb_name", "CDRH3"])
meta_len = pd.concat([test_meta_len, train_meta_len], ignore_index=True).drop_duplicates("pdb_name")
meta_len["cdrh3_len"] = meta_len["CDRH3"].astype(str).str.len()
pdb_to_len = meta_len.set_index("pdb_name")["cdrh3_len"]

pair_len_df = (
    gt_pairs_all.query("status == 'ok'")
    .assign(
        len_a=lambda d: d["pdb_name_a"].map(pdb_to_len),
        len_b=lambda d: d["pdb_name_b"].map(pdb_to_len),
    )
    .dropna(subset=[PRIMARY_PAIR_METRIC, "len_a", "len_b"])
)
pair_len_df["len_a"] = pair_len_df["len_a"].astype(int)
pair_len_df["len_b"] = pair_len_df["len_b"].astype(int)
pair_len_df["abs_len_diff"] = (pair_len_df["len_a"] - pair_len_df["len_b"]).abs()
pair_len_df["same_cdrh3_len"] = pair_len_df["len_a"] == pair_len_df["len_b"]
pair_len_df["mean_len"] = (pair_len_df["len_a"] + pair_len_df["len_b"]) / 2.0

len_vals = pair_len_df[["len_a", "len_b"]].stack()
q05, q95 = len_vals.quantile([0.05, 0.95])
CDRH3_LEN_BIN_EDGES = sorted(set([
    max(1, int(np.floor(q05)) - 1),
    8, 11, 14, 17,
    int(np.ceil(q95)) + 1,29
]))
pair_len_df["mean_len_bin"] = pd.cut(
    pair_len_df["mean_len"], bins=CDRH3_LEN_BIN_EDGES, right=False, include_lowest=True,
)
pair_len_df["len_a_bin"] = pd.cut(
    pair_len_df["len_a"], bins=CDRH3_LEN_BIN_EDGES, right=False, include_lowest=True,
)
pair_len_df["len_b_bin"] = pd.cut(
    pair_len_df["len_b"], bins=CDRH3_LEN_BIN_EDGES, right=False, include_lowest=True,
)

pair_len_df = pair_len_df.merge(gt_cluster_all, on="cluster_ids", how="left")
pair_len_df["abs_len_diff_cap"] = pair_len_df["abs_len_diff"].clip(upper=6)
pair_len_df["abs_len_diff_label"] = pair_len_df["abs_len_diff_cap"].astype(int).astype(str)
pair_len_df.loc[pair_len_df["abs_len_diff"] > 6, "abs_len_diff_label"] = "6+"

n_ok_pairs = len(gt_pairs_all.query("status == 'ok'"))
missing_meta = n_ok_pairs - len(pair_len_df)
print(f"Pair rows (ok): {n_ok_pairs:,} -> with lengths: {len(pair_len_df):,} (dropped {missing_meta:,})")
print(f"  test pairs: {(pair_len_df['split'] == 'test').sum():,}  train pairs: {(pair_len_df['split'] == 'train').sum():,}")
print(f"CDRH3 length bin edges: {CDRH3_LEN_BIN_EDGES}")
print(
    "Same vs mixed length — median H_cdr3:",
    pair_len_df.groupby('same_cdrh3_len')[PRIMARY_PAIR_METRIC].median().to_dict(),
)


In [ ]:
# --- 1) H_cdr3 vs |Δ CDRH3 length| ---
diff_order = [str(i) for i in range(7)] + ["6+"]
diff_order = [x for x in diff_order if x in pair_len_df["abs_len_diff_label"].unique()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=pair_len_df,
    x="abs_len_diff_label",
    y=PRIMARY_PAIR_METRIC,
    order=diff_order,
    showfliers=False,
    color="#5F7D95",
    ax=axes[0],
)
axes[0].set_xlabel("|CDRH3 length A − length B| (capped label at 6+)")
axes[0].set_ylabel("Pairwise H_cdr3 (Å)")
axes[0].set_title("GT intra-cluster H_cdr3 vs CDRH3 length mismatch (train + test)")

# by cluster size bucket (n_members of cluster)
pair_len_df["size_bucket"] = pd.cut(
    pair_len_df["n_members"],
    bins=[1, 3, 10, 25, 10_000],
    labels=["2–3", "4–10", "11–25", "26+"],
)
sns.boxplot(
    data=pair_len_df,
    x="abs_len_diff_label",
    y=PRIMARY_PAIR_METRIC,
    hue="size_bucket",
    order=diff_order,
    showfliers=False,
    ax=axes[1],
)
axes[1].set_xlabel("|Δ CDRH3 length|")
axes[1].set_ylabel("Pairwise H_cdr3 (Å)")
axes[1].set_title("Colored by cluster size (n_members)")
axes[1].legend(title="Cluster size", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
fig.savefig(FIG_DIR / "gt_H_cdr3_vs_cdrh3_len_diff_train_test.png", dpi=150, bbox_inches="tight")
plt.show()

# --- 2) Same-length vs mixed-length pairs ---
same_len_order = ["Same CDRH3 length", "Different CDRH3 length"]
same_len_palette = {"Same CDRH3 length": "#6B9080", "Different CDRH3 length": "#A65D57"}
plot_same_len = pair_len_df.assign(
    same_cdrh3_len_label=pair_len_df["same_cdrh3_len"].map(
        {True: "Same CDRH3 length", False: "Different CDRH3 length"}
    )
)
fig, ax = plt.subplots(figsize=(6, 5))
sns.violinplot(
    data=plot_same_len,
    x="same_cdrh3_len_label",
    y=PRIMARY_PAIR_METRIC,
    order=same_len_order,
    cut=0,
    inner="quartile",
    palette=same_len_palette,
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("Pairwise H_cdr3 (Å)")
ax.set_title("Same-length vs mixed-length pairs within cluster (train + test)")
plt.tight_layout()
fig.savefig(FIG_DIR / "gt_H_cdr3_same_vs_mixed_cdrh3_len_train_test.png", dpi=150, bbox_inches="tight")
plt.show()

summary_same = pair_len_df.groupby("same_cdrh3_len").agg(
    n_pairs=(PRIMARY_PAIR_METRIC, "count"),
    median_H_cdr3=(PRIMARY_PAIR_METRIC, "median"),
    mean_H_cdr3=(PRIMARY_PAIR_METRIC, "mean"),
)
display(summary_same)


In [ ]:
# --- 3) Cluster × mean pair CDRH3 length bin (sparse heatmap) ---
top_clusters = (
    gt_cluster_all.sort_values("n_members", ascending=False)
    .head(TOP_CLUSTERS_FOR_HEATMAP)["cluster_ids"]
    .tolist()
)
hm_df = pair_len_df[pair_len_df["cluster_ids"].isin(top_clusters)].copy()

agg = (
    hm_df.groupby(["cluster_ids", "mean_len_bin"], observed=True)[PRIMARY_PAIR_METRIC]
    .agg(mean_rmsd="mean", n_pairs="count")
    .reset_index()
)
mean_pivot = agg.pivot(index="cluster_ids", columns="mean_len_bin", values="mean_rmsd")
count_pivot = agg.pivot(index="cluster_ids", columns="mean_len_bin", values="n_pairs")
mask = (count_pivot.reindex_like(mean_pivot) < MIN_PAIRS_HEATMAP) | mean_pivot.isna()

annot = mean_pivot.copy().astype(object)
for r in annot.index:
    for c in annot.columns:
        val = mean_pivot.loc[r, c] if c in mean_pivot.columns else np.nan
        n = count_pivot.loc[r, c] if (r in count_pivot.index and c in count_pivot.columns) else np.nan
        if pd.isna(val):
            annot.loc[r, c] = ""
        else:
            annot.loc[r, c] = f"{val:.1f}\n(n={int(n)})"

cluster_order = (
    gt_cluster_all.set_index("cluster_ids").loc[top_clusters, "n_members"].sort_values(ascending=False).index
)
mean_pivot = mean_pivot.reindex(cluster_order)
mask = mask.reindex_like(mean_pivot)
annot = annot.reindex_like(mean_pivot)

fig_h = max(8, 0.35 * len(mean_pivot))
fig, ax = plt.subplots(figsize=(12, fig_h))
sns.heatmap(
    mean_pivot,
    mask=mask,
    annot=annot,
    fmt="",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Mean pairwise CDRH3 RMSD (Å)"},
    ax=ax,
)
ax.set_title(
    f"Mean GT CDRH3 RMSD by cluster stratified by mean-pair CDRH3 length (largest {TOP_CLUSTERS_FOR_HEATMAP} clusters; "
    f"mask n<{MIN_PAIRS_HEATMAP})"
)
ax.set_xlabel("Mean CDRH3 length bin (pair average)")
ax.set_ylabel("cluster_ids")
plt.tight_layout()
fig.savefig(FIG_DIR / "gt_H_cdr3_heatmap_cluster_x_mean_len_train_test.png", dpi=150, bbox_inches="tight")
plt.show()
pct_masked = float(mask.to_numpy().sum()) / mask.size * 100
print(f"Cluster×length heatmap: {pct_masked:.1f}% cells masked (n < {MIN_PAIRS_HEATMAP})")

# --- 4) Global length × length heatmap (pooled over clusters) ---
len_agg = (
    pair_len_df.groupby(["len_a_bin", "len_b_bin"], observed=True)[PRIMARY_PAIR_METRIC]
    .agg(mean_rmsd="mean", n_pairs="count")
    .reset_index()
)
len_mean = len_agg.pivot(index="len_a_bin", columns="len_b_bin", values="mean_rmsd")
len_count = len_agg.pivot(index="len_a_bin", columns="len_b_bin", values="n_pairs")
len_mask = (len_count.reindex_like(len_mean) < MIN_PAIRS_HEATMAP) | len_mean.isna()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    len_mean,
    mask=len_mask,
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Mean pairwise CDRH3 RMSD (Å)"},
    ax=ax,
)
#ax.set_title(f"Pooled length × length mean H_cdr3 (train + test; mask n<{MIN_PAIRS_HEATMAP})")
ax.set_xlabel("CDRH3 A length")
ax.set_ylabel("CDRH3 B length")
plt.tight_layout()
fig.savefig(FIG_DIR / "gt_H_cdr3_heatmap_len_x_len_train_test.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
pair_len_df.len_b.max()


In [ ]:
# --- 5) Cluster-level: length heterogeneity vs mean pairwise H_cdr3 ---
gt_cluster_summary = pd.read_csv(OUT_DIR / "cluster_gt_pairwise_rmsd.csv")[[
    "cluster_ids", "n_members", "mean_pairwise_H_cdr3"
]]
gt_cluster_summary_train = pd.read_csv(TRAIN_OUT / "cluster_gt_train_pairwise_rmsd.csv")[[
    "cluster_ids", "n_members", "mean_pairwise_H_cdr3"
]]
gt_intra = pd.concat([gt_cluster_summary, gt_cluster_summary_train], ignore_index=True)
gt_intra = (
    gt_intra.groupby("cluster_ids", as_index=False)
    .agg(n_members=("n_members", "max"), mean_pairwise_H_cdr3=("mean_pairwise_H_cdr3", "mean"))
)

# per-structure lengths within each cluster (train + test meta)
struct_len = meta_len.merge(
    pd.concat([
        pd.read_csv(TEST_META_PATH, usecols=["pdb_name", "cluster_ids"]),
        pd.read_csv(TRAIN_META_PATH, usecols=["pdb_name", "cluster_ids"]),
    ], ignore_index=True),
    on="pdb_name",
    how="inner",
)
len_hetero = struct_len.groupby("cluster_ids")["cdrh3_len"].agg(
    n_structures="count",
    n_unique_lengths="nunique",
    std_cdrh3_len="std",
    range_cdrh3_len=lambda s: s.max() - s.min(),
).reset_index()

cluster_scatter = gt_intra.merge(len_hetero, on="cluster_ids", how="inner")
cluster_scatter = cluster_scatter[cluster_scatter["n_members"] >= 2]

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(
    cluster_scatter["range_cdrh3_len"],
    cluster_scatter["mean_pairwise_H_cdr3"],
    s=np.clip(cluster_scatter["n_members"] * 3, 20, 200),
    alpha=0.55,
    c="#5F7D95",
    edgecolors="0.3",
    linewidths=0.4,
)
ax.set_xlabel("CDRH3 length range within cluster (max − min)")
ax.set_ylabel("Mean pairwise GT H_cdr3 (Å)")
ax.set_title("Cluster-level length spread vs intra-cluster H_cdr3 (train + test clusters)")

r, p = stats.spearmanr(
    cluster_scatter["range_cdrh3_len"],
    cluster_scatter["mean_pairwise_H_cdr3"],
)
ax.text(0.03, 0.97, f"Spearman ρ = {r:.3f} (p = {p:.2g})", transform=ax.transAxes, va="top")
plt.tight_layout()
fig.savefig(FIG_DIR / "gt_cluster_len_range_vs_mean_H_cdr3_train_test.png", dpi=150, bbox_inches="tight")
plt.show()
display(cluster_scatter.nlargest(10, "mean_pairwise_H_cdr3")[[
    "cluster_ids", "n_members", "range_cdrh3_len", "n_unique_lengths", "mean_pairwise_H_cdr3"
]])
